In [1]:
import os
import torch
from PIL import Image
from tqdm import tqdm
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import f1_score, confusion_matrix
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [3]:

torch.cuda.empty_cache()
print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU Memory Cached: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

GPU Memory Allocated: 0.00 GB
GPU Memory Cached: 0.00 GB


In [5]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Config (unchanged)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 20
NUM_CLASSES = 3
TRAIN_DIR = "../split_data/train"
TEST_DIR = "../split_data/test"
GRAD_ACCUMULATION_STEPS = 8
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUMULATION_STEPS

# Data Augmentation (unchanged)
def get_transforms(train=True):
    if train:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.RandomRotation(15),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])

# Dataset (unchanged)
class ChestXrayDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        self.images = self._load_images()

    def _load_images(self):
        images = []
        for cls in self.classes:
            cls_dir = os.path.join(self.root_dir, cls)
            for img_name in os.listdir(cls_dir):
                if img_name.endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(cls_dir, img_name)
                    images.append((img_path, self.class_to_idx[cls]))
        return images

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path, label = self.images[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

# Model - Changed to ConvNeXt Base
class ConvNeXtBaseTransformer(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        # Load pre-trained ConvNeXt Base model
        base_model = models.convnext_base(weights='IMAGENET1K_V1')
        
        # Extract features up to last convolutional layer
        self.cnn = nn.Sequential(*list(base_model.children())[:-2])
        
        # Get ConvNeXt Base feature dimension (1024 for convnext_base)
        convnext_feature_dim = 1024
        
        # Transformer Head (unchanged architecture)
        self.transformer_head = TransformerHead(
            input_dim=convnext_feature_dim,
            hidden_dim=512,  # Same hidden dim as tiny version
            num_classes=num_classes
        )
        
    def forward(self, x):
        # CNN Feature Extraction
        x = self.cnn(x)  # (B, 1024, 7, 7) for ConvNeXt base
        
        # Global Average Pooling
        x = x.mean(dim=[2, 3])  # (B, 1024)
        
        # Transformer Processing
        return self.transformer_head(x)

# Transformer Head (unchanged)
class TransformerHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, nhead=8, num_layers=2):
        super().__init__()
        self.projection = nn.Linear(input_dim, hidden_dim)
        
        # Transformer Encoder
        encoder_layer = TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=nhead,
            dim_feedforward=hidden_dim*4,
            dropout=0.2,
            activation='gelu',
            batch_first=True
        )
        self.transformer = TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, num_classes))
        
    def forward(self, x):
        x = self.projection(x).unsqueeze(1)  # (B, 1, hidden_dim)
        x = self.transformer(x)
        x = x.mean(dim=1)  # Pool sequence
        return self.classifier(x)

# Training Utilities (unchanged)
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    optimizer.zero_grad()
    
    for batch_idx, (inputs, labels) in enumerate(tqdm(loader, desc="Training")):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        outputs = model(inputs)
        loss = criterion(outputs, labels) / GRAD_ACCUMULATION_STEPS
        loss.backward()
        
        running_loss += loss.item() * GRAD_ACCUMULATION_STEPS
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        if (batch_idx + 1) % GRAD_ACCUMULATION_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()
    
    # Final step if remaining
    if (batch_idx + 1) % GRAD_ACCUMULATION_STEPS != 0:
        optimizer.step()
        optimizer.zero_grad()
    
    acc = correct / total
    f1 = f1_score(all_labels, all_preds, average='macro')
    return running_loss / len(loader), acc, f1

def evaluate(model, loader, criterion, return_predictions=False):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc="Evaluating"):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    acc = correct / total
    f1 = f1_score(all_labels, all_preds, average='macro')
    
    if return_predictions:
        return running_loss / len(loader), acc, f1, all_preds, all_labels
    return running_loss / len(loader), acc, f1

def plot_confusion_matrix(true_labels, pred_labels, class_names, filename='confusion_matrix_convnext_base.png'):
    cm = confusion_matrix(true_labels, pred_labels)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.savefig(filename, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrix saved to {filename}")

# Main Training Loop (unchanged except model name)
def main():
    # Data Loaders
    train_dataset = ChestXrayDataset(TRAIN_DIR, get_transforms(train=True))
    val_dataset = ChestXrayDataset(TEST_DIR, get_transforms(train=False))
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # Model - Now using ConvNeXt Base
    model = ConvNeXtBaseTransformer(num_classes=NUM_CLASSES).to(DEVICE)
    
    # Loss and Optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    
    # Training Tracking
    best_f1 = 0.0
    history = {'train_loss': [], 'train_acc': [], 'train_f1': [],
               'val_loss': [], 'val_acc': [], 'val_f1': []}
    
    print(f"Training ConvNeXt Base-Transformer Hybrid on {DEVICE}")
    print(f"Effective batch size: {EFFECTIVE_BATCH_SIZE}")
    
    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        
        # Train
        train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
        
        # Validate
        val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion)
        
        # Update history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_f1'].append(train_f1)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        
        # Scheduler step
        scheduler.step(val_f1)
        
        print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")
        
        # Save best model
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), 'best_convnext_base_model.pth')
            print(f"Saved new best model with F1: {best_f1:.4f}")
    
    # Final Evaluation
    print("\n=== Final Evaluation ===")
    model.load_state_dict(torch.load('best_convnext_base_model.pth'))
    _, val_acc, val_f1, all_preds, all_labels = evaluate(
        model, val_loader, criterion, return_predictions=True
    )
    
    print(f"Best Model Metrics - Acc: {val_acc:.4f}, F1: {val_f1:.4f}")
    
    # Confusion Matrix
    plot_confusion_matrix(
        all_labels, all_preds, 
        class_names=train_dataset.classes,
        filename='convnext_base_transformer_cm.png'
    )
    
    # Plot Training Curves
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.plot(history['train_loss'], label='Train')
    plt.plot(history['val_loss'], label='Val')
    plt.title('Loss Curve')
    plt.legend()
    
    plt.subplot(1, 3, 2)
    plt.plot(history['train_acc'], label='Train')
    plt.plot(history['val_acc'], label='Val')
    plt.title('Accuracy Curve')
    plt.legend()
    
    plt.subplot(1, 3, 3)
    plt.plot(history['train_f1'], label='Train')
    plt.plot(history['val_f1'], label='Val')
    plt.title('F1 Score Curve')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig('convnext_base_training_curves.png')
    plt.close()
    print("Training curves saved to convnext_base_training_curves.png")



In [6]:
if __name__ == "__main__":
    main()

Training ConvNeXt Base-Transformer Hybrid on cuda
Effective batch size: 64

Epoch 1/20


Evaluating: 100%|████████████████████████████████████████████████████████████████████| 172/172 [01:11<00:00,  2.40it/s]


Train Loss: 0.3235 | Acc: 0.8854 | F1: 0.8862
Val Loss: 0.1751 | Acc: 0.9462 | F1: 0.9468
Saved new best model with F1: 0.9468

Epoch 2/20


Evaluating: 100%|████████████████████████████████████████████████████████████████████| 172/172 [01:00<00:00,  2.84it/s]


Train Loss: 0.1643 | Acc: 0.9461 | F1: 0.9466
Val Loss: 0.1476 | Acc: 0.9535 | F1: 0.9541
Saved new best model with F1: 0.9541

Epoch 3/20


...

Train Loss: 0.0100 | Acc: 0.9971 | F1: 0.9971
Val Loss: 0.2329 | Acc: 0.9629 | F1: 0.9633

Epoch 20/20


Evaluating: 100%|████████████████████████████████████████████████████████████████████| 172/172 [01:07<00:00,  2.55it/s]


Train Loss: 0.0150 | Acc: 0.9951 | F1: 0.9951
Val Loss: 0.2483 | Acc: 0.9615 | F1: 0.9619

=== Final Evaluation ===


Evaluating: 100%|████████████████████████████████████████████████████████████████████| 172/172 [01:08<00:00,  2.52it/s]


Best Model Metrics - Acc: 0.9629, F1: 0.9633
Confusion matrix saved to convnext_base_transformer_cm.png
Training curves saved to convnext_base_training_curves.png
